# Go2 Course Homework: Teaching Colab Template

Welcome to Assignment 1.

It assumes that the readable homework package is stored in a course GitHub repo: https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git
This version keeps the important
source files visible as normal `.py` / `.json` files.

This release provides baseline intentionally only
learns `{stand, +vx}`. Students then extend the same pipeline so the robot can
track forward / backward motion, lateral motion, and yaw commands.

`stage_1` and `stage_2` are internal training curriculum stages, not two
separate homework assignments.

Recommended lecture / demo usage:
1. run setup
2. inspect the environment
3. show the key files
4. train or restore a checkpoint
5. run the public benchmark and discuss the axis-wise metrics


## 1. Configure repository URLs

Set `COURSE_REPO_URL` to the GitHub repository that contains this readable
homework package. The repo should contain:
- `train.py`
- `test_policy.py`
- `generate_public_rollout.py`
- `public_eval.py`
- `go2_pg_env/`
- `configs/course_config.json`

Important Notice: Since this assignment runs in Google Colab, you should not treat the Colab runtime as permanent storage. Before you start, create your own GitHub repository by either forking this repo or creating a new repo based on it, then change the notebook’s course_repo_url to point to your own repository instead of the course repo. From that point on, all of your work should be done in the copy cloned from your own repository. After making any changes in Colab, you should regularly save them back to GitHub using git so that your code is stored safely and remains reproducible. In short: replace the repo URL with your own, do all development in that cloned copy, and push your changes to GitHub regularly rather than relying on Colab to keep your files.

In [1]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path("/content/go2_course_repo")

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

PLAYGROUND_DIR = Path("/content/mujoco_playground")
UNITREE_DIR = Path("/content/unitree_mujoco")

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


Running inside Colab.


## 2. Install system packages and clone repositories

In [2]:
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
!python -m pip install -q -U pip setuptools wheel "jedi>=0.16"

import importlib.util
if importlib.util.find_spec("playground") is not None:
    !python -m pip uninstall -y playground

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / "configs" / "colab_requirements.txt"}

%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .

%cd {COURSE_REPO_DIR}
print("Setup finished. Some Colab dependency warnings can usually be ignored if the later checks pass.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 119.9 MB/s eta 0:00:00
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git /content/go2_course_repo
+ git clone https://github.com/deepmind/mujoco_menagerie.git /content/mujoco_playground/mujoco_playground/external_deps/mujoco_menagerie
+ git -C /content/mujoco_playground/mujoco_playground/external_deps/

## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

Use the repo-side helper script so the notebook stays aligned with the code
students will download and edit.


In [3]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


/content/go2_course_repo
Copied 16 assets into /content/go2_course_repo/go2_pg_env/xmls/assets


## 4. Inspect the environment before training

Run this before creating the Colab runtime override file. The default course
config already contains the released forward-only baseline setup.


In [4]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_1


/content/go2_course_repo
E0417 23:17:09.179764    1215 ptx_compiler_helpers.cc:132] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_1",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_

## 5. Read the most important files

Students should focus on:
1. `go2_pg_env/joystick.py`
2. `configs/course_config.json`
3. `benchmark_specs.py`
4. `public_eval.py`
5. `train.py`


In [ ]:
!sed -n '1,260p' go2_pg_env/joystick.py


"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import math
import numpy as np

from mujoco_playground._src import mjx_env



In [ ]:
!sed -n '1,220p' train.py
!printf '\n===== configs/course_config.json =====\n'
!sed -n '1,240p' configs/course_config.json


#!/usr/bin/env python3
"""Train the course Go2 locomotion policy with PPO.

Design goals
------------
1. Keep the file small and readable.
2. Preserve the original two-stage training behavior.
3. Make every output artifact explicit:
   - progress.json
   - summary.json
   - resolved_config.json
   - best_checkpoint/manifest.json
"""

from __future__ import annotations

import argparse
import functools
import json
import os
import time
from pathlib import Path
from typing import Any

from course_common import (
    DEFAULT_CONFIG_PATH,
    apply_stage_config,
    build_env_overrides,
    detect_gpu_name,
    ensure_environment_available,
    export_selected_checkpoint,
    get_ppo_config,
    lazy_import_stack,
    load_json,
    resolve_latest_checkpoint_dir,
    save_json,
    set_runtime_env,
    stage_sequence,
    to_jsonable,
)


ROOT = Path(__file__).resolve().parent


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add

In [ ]:
!sed -n '1,220p' benchmark_specs.py
!printf '\n===== public_eval.py =====\n'
!sed -n '1,240p' public_eval.py


#!/usr/bin/env python3
"""Deterministic command scripts used for demo videos and public benchmark runs."""

from __future__ import annotations

from typing import Any

import numpy as np


def build_demo_segments(config: dict[str, Any]) -> list[list[float]]:
    """Return an easy-to-read command script for demo videos.

    The demo is intentionally human-readable:
    stand -> forward -> turn -> strafe -> combined motion -> stand.
    """
    demo_cfg = config["demo_rollout"]
    if "segments" in demo_cfg and demo_cfg["segments"]:
        return [[float(x) for x in segment] for segment in demo_cfg["segments"]]
    return [
        [0.0, 0.0, 0.0],
        [0.35, 0.0, 0.0],
        [0.0, 0.0, 0.45],
        [0.0, 0.15, 0.0],
        [0.30, -0.10, -0.30],
        [0.0, 0.0, 0.0],
    ]


def public_command_script(safe_ranges: dict[str, list[float]], episode_idx: int) -> list[list[float]]:
    """Return the deterministic command schedule for one public benchmark episode."""
    vx_min, v

## 6. Define a Colab-friendly runtime config

Write a small `runtime_overrides` block on top of the base course config.
This keeps the released homework settings intact while still letting Colab
use a smaller or larger training profile.


In [5]:
import json

runtime_overrides = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_overrides
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


wrote /content/go2_course_repo/configs/colab_runtime_config.json


## 7. Dry-run the training config

In [6]:
!python train.py --config configs/colab_runtime_config.json --dry-run

{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_hidden_layer_sizes": [
      256,
      256,
      128
    ]
  },
  

## 8. Run training

This command trains the **released forward-only baseline**.
Students should later modify `go2_pg_env/joystick.py` and possibly
`configs/course_config.json`, then rerun training and compare benchmark
metrics.


In [ ]:
%cd {COURSE_REPO_DIR}

!python train.py \
  --config configs/colab_runtime_config.json \
  --stage both \
  --output-dir artifacts/run_baseline


/content/go2_course_repo
[run] output_dir=/content/go2_course_repo/artifacts/run_baseline
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5
E0417 23:18:46.981951    3501 ptx_compiler_helpers.cc:132] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
[stage_1] steps=0 eval_reward=0.005
[stage_1] steps=3276800 eval_reward=2.824
[stage_1] steps=6553600 eval_reward=22.412
[stage_1] steps=9830400 eval_reward=26.912


## 9. Restore a checkpoint and render a deterministic demo

The demo script still contains turn / strafe / combined segments on purpose.
A released forward-only baseline is expected to look okay on standing / forward
segments and much worse on the others until students extend the command sampler.


In [ ]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_baseline" / "best_checkpoint"
DEMO_DIR = COURSE_REPO_DIR / "artifacts" / "demo_bundle"

%cd {COURSE_REPO_DIR}

!python test_policy.py \
  --config configs/colab_runtime_config.json \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {DEMO_DIR}


/content/go2_course_repo
E0417 22:25:45.872456    7252 ptx_compiler_helpers.cc:132] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
100% 751/751 [01:49<00:00,  6.86it/s]
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_baseline/best_checkpoint",
  "num_steps": 750,
  "metrics": {
    "velocity_tracking_error": 0.07531354576349258,
    "yaw_tracking_error": 0.09578517824411392,
    "fall_rate": 0.0,
    "energy_proxy": 10.108941078186035,
    "foot_slip_proxy": 0.09944720566272736
  },
  "normalized_scores": {
    "velocity_tracking_error": 1.0,
    "yaw_tracking_error": 1.0,
    "fall_rate": 1.0,
    "energy_prox

## 10. Generate the public benchmark rollout

The public benchmark contains four deterministic cases:
1. forward / backward `vx`
2. lateral `vy`
3. yaw
4. combined `vx + vy + yaw`

`public_eval.py` reports axis-wise errors so students can analyze exactly
which capabilities improved and which did not.


In [ ]:
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle"

%cd {COURSE_REPO_DIR}

!python generate_public_rollout.py \
  --config configs/colab_runtime_config.json \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {PUBLIC_DIR} \
  --num-episodes 4 \
  --render-first-episode

!python public_eval.py \
  --config configs/colab_runtime_config.json \
  --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} \
  --output-json {PUBLIC_DIR / "public_eval.json"}


/content/go2_course_repo
E0417 22:30:16.720227    8950 ptx_compiler_helpers.cc:132] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
100% 1501/1501 [03:33<00:00,  7.04it/s]
{
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_baseline/best_checkpoint",
  "stage_name": "stage_2",
  "num_episodes": 4,
  "episode_length_steps": 1500,
  "episode_lengths_realized": [
    1500,
    1500,
    1500,
    1500
  ],
  "rollout_npz": "/content/go2_course_repo/artifacts/public_eval_bundle/rollout_public_eval.npz",
  "video_path": "/content/go2_course_repo/artifacts/public_eval_bundle/public_eval_episode0.mp4"
}
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "num_steps": 6000,
  "num_episodes": 4,
  